# C12-classical-models — Practice p21 — Solution


The split, candidate order, folds, estimator state, selection rule, and label-free k-means audit are all explicit in the returned ledger.


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

P21_FAMILY_ORDER = (
    "logistic_regression", "svm", "decision_tree", "random_forest"
)
P21_CANDIDATE_NAMES = {
    "logistic_regression": ("logistic_C_0.25", "logistic_C_1.0"),
    "svm": ("svm_linear_C_1.0", "svm_rbf_C_1.0_gamma_1.0"),
    "decision_tree": ("tree_depth_2", "tree_depth_4"),
    "random_forest": ("forest_depth_3", "forest_depth_None"),
}
P21_KMEANS_SEEDS = np.array([20260804, 20260805, 20260806], dtype=np.int64)
P21_PRIMARY_KMEANS_INDEX = 0

X_p21, y_p21 = make_moons(n_samples=240, noise=0.18, random_state=20260804)
X_p21 = X_p21.astype(np.float64)
y_p21 = y_p21.astype(np.int64)


def benchmark_classical_models(X, y):
    indices = np.arange(X.shape[0], dtype=np.int64)
    X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
        X, y, indices, test_size=0.25, train_size=None, random_state=20260804,
        shuffle=True, stratify=y
    )
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=20260804)
    scaler = lambda: StandardScaler(copy=True, with_mean=True, with_std=True)
    logistic_candidates = (
        make_pipeline(scaler(), LogisticRegression(C=0.25, penalty="l2", fit_intercept=True,
            solver="lbfgs", dual=False, tol=1e-4, max_iter=1000, class_weight=None,
            random_state=20260804)),
        make_pipeline(scaler(), LogisticRegression(C=1.0, penalty="l2", fit_intercept=True,
            solver="lbfgs", dual=False, tol=1e-4, max_iter=1000, class_weight=None,
            random_state=20260804)),
    )
    svm_candidates = (
        make_pipeline(scaler(), SVC(kernel="linear", C=1.0, gamma=1.0, degree=3, coef0=0.0,
            shrinking=True, probability=False, tol=1e-3, class_weight=None, max_iter=-1,
            decision_function_shape="ovr", break_ties=False, random_state=20260804)),
        make_pipeline(scaler(), SVC(kernel="rbf", C=1.0, gamma=1.0, degree=3, coef0=0.0,
            shrinking=True, probability=False, tol=1e-3, class_weight=None, max_iter=-1,
            decision_function_shape="ovr", break_ties=False, random_state=20260804)),
    )
    tree_candidates = tuple(DecisionTreeClassifier(criterion="gini", splitter="best",
        max_depth=depth, min_samples_split=2, min_samples_leaf=1, max_features=None,
        max_leaf_nodes=None, min_impurity_decrease=0.0, ccp_alpha=0.0,
        random_state=20260804) for depth in (2,4))
    forest_candidates = tuple(RandomForestClassifier(n_estimators=40, criterion="gini",
        max_depth=depth, min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
        bootstrap=True, oob_score=False, warm_start=False, max_samples=None,
        class_weight=None, max_leaf_nodes=None, min_impurity_decrease=0.0, ccp_alpha=0.0,
        n_jobs=1, random_state=20260804) for depth in (3,None))
    specifications = (
        ("logistic_regression", P21_CANDIDATE_NAMES["logistic_regression"], logistic_candidates,
         ({"C":0.25},{"C":1.0}), "probability"),
        ("svm", P21_CANDIDATE_NAMES["svm"], svm_candidates,
         ({"kernel":"linear","C":1.0,"gamma":1.0},{"kernel":"rbf","C":1.0,"gamma":1.0}), "decision_function"),
        ("decision_tree", P21_CANDIDATE_NAMES["decision_tree"], tree_candidates,
         ({"max_depth":2},{"max_depth":4}), "probability"),
        ("random_forest", P21_CANDIDATE_NAMES["random_forest"], forest_candidates,
         ({"max_depth":3},{"max_depth":None}), "probability"),
    )
    supervised = {}
    for family, names, candidates, parameter_options, score_kind in specifications:
        fold_scores = np.vstack([cross_val_score(candidate, X_train, y_train, cv=cv, scoring="accuracy")
                                 for candidate in candidates]).astype(np.float64)
        mean_scores = fold_scores.mean(axis=1).astype(np.float64)
        selected_index = int(np.argmax(mean_scores))
        estimator = candidates[selected_index].fit(X_train, y_train)
        test_predictions = estimator.predict(X_test).astype(np.int64)
        if score_kind == "decision_function":
            test_scores = estimator.decision_function(X_test).astype(np.float64)
        else:
            test_scores = estimator.predict_proba(X_test)[:,1].astype(np.float64)
        supervised[family] = {"candidate_names": names, "fold_scores": fold_scores,
            "mean_scores": mean_scores, "selected_index": selected_index,
            "selected_params": parameter_options[selected_index], "estimator": estimator,
            "test_predictions": test_predictions, "test_scores": test_scores,
            "score_kind": score_kind,
            "test_accuracy": float(np.mean(test_predictions == y_test))}
    kmeans_scaler = StandardScaler(copy=True, with_mean=True, with_std=True).fit(X_train)
    scaled_train = kmeans_scaler.transform(X_train)
    scaled_test = kmeans_scaler.transform(X_test)
    stability_seeds = P21_KMEANS_SEEDS.copy()
    models = tuple(KMeans(n_clusters=2, init="k-means++", n_init=10, max_iter=300,
        tol=1e-4, copy_x=True, algorithm="lloyd", random_state=int(seed)).fit(scaled_train)
        for seed in stability_seeds)
    labels = np.vstack([model.labels_ for model in models]).astype(np.int64)
    co_clustering = labels[:,:,None] == labels[:,None,:]
    upper = np.triu_indices(X_train.shape[0], k=1)
    agreements = np.array([np.mean(matrix[upper] == co_clustering[P21_PRIMARY_KMEANS_INDEX][upper])
                           for matrix in co_clustering], dtype=np.float64)
    split = {"train_indices":train_indices.astype(np.int64), "test_indices":test_indices.astype(np.int64),
             "X_train":X_train.astype(np.float64), "X_test":X_test.astype(np.float64),
             "y_train":y_train.astype(np.int64), "y_test":y_test.astype(np.int64)}
    kmeans = {"scaler":kmeans_scaler, "models":models, "stability_seeds":stability_seeds,
              "labels":labels, "co_clustering":co_clustering,
              "agreements_to_primary":agreements,
              "train_distances":models[P21_PRIMARY_KMEANS_INDEX].transform(scaled_train).astype(np.float64),
              "test_distances":models[P21_PRIMARY_KMEANS_INDEX].transform(scaled_test).astype(np.float64),
              "inertia":float(models[P21_PRIMARY_KMEANS_INDEX].inertia_)}
    return {"split":split, "supervised":supervised, "kmeans":kmeans}


benchmark_p21 = benchmark_classical_models(X_p21, y_p21)
comparison_axes_p21 = '''Logistic regression and SVM are supervised scaled linear or kernel models with BCE/probability and hinge/decision-score outputs respectively; trees and forests are supervised axis-aligned nonlinear models, with the forest trading direct interpretability for variance reduction; k-means is unsupervised centroid geometry with WCSS and distances, never class predictions. Scaling is fold-local or training-only. Coefficients, rules, forest aggregates, and centroids offer different interpretations. Four-fold training CV selects supervised complexity, the test set is used once, and co-clustering stability audits k-means without labels.'''


### Answer check


In [ ]:
ATOL = 1e-10
RTOL = 1e-8
assert set(benchmark_p21) == {"split","supervised","kmeans"}
assert set(benchmark_p21["supervised"]) == {"logistic_regression","svm","decision_tree","random_forest"}
expected_selected_p21 = {"logistic_regression":1,"svm":1,"decision_tree":0,"random_forest":1}
for family_p21, selected_p21 in expected_selected_p21.items():
    result_family_p21 = benchmark_p21["supervised"][family_p21]
    assert result_family_p21["fold_scores"].shape == (2,4)
    assert result_family_p21["selected_index"] == selected_p21
    assert result_family_p21["test_predictions"].shape == (60,)
assert benchmark_p21["split"]["train_indices"].shape == (180,)
assert benchmark_p21["split"]["test_indices"].shape == (60,)
assert benchmark_p21["kmeans"]["train_distances"].shape == (180,2)
assert benchmark_p21["kmeans"]["test_distances"].shape == (60,2)
assert np.isclose(benchmark_p21["kmeans"]["inertia"], 161.25975964790527, atol=ATOL, rtol=RTOL)
assert np.allclose(benchmark_p21["kmeans"]["agreements_to_primary"], [1.0,1.0,1.0], atol=ATOL, rtol=RTOL)
assert isinstance(comparison_axes_p21, str) and "never" in comparison_axes_p21
